In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

file_path="D:/Wia/Code(My)/data(2).csv"
import pandas as pd
df = pd.read_csv(file_path)

df.head()


,node_id,proto_mean,seq_mean,stddev_mean,N_IN_Conn_P_SrcIP_mean,min_mean,state_number_mean,mean_mean,N_IN_Conn_P_DstIP_mean,drate_mean,srate_mean,max_mean,attack
0,192.168.100.1->192.168.100.147,0,4521.000000,0.0,6.000000,0.000055,2,0.000055,1,0.0,0.0,0.000055,0
1,192.168.100.1->192.168.100.148,0,4280.000000,0.0,6.000000,0.000058,2,0.000058,1,0.0,0.0,0.000058,0
2,192.168.100.1->192.168.100.149,0,5412.000000,0.0,6.000000,0.000111,2,0.000111,1,0.0,0.0,0.000111,0
3,192.168.100.1->192.168.100.150,0,4287.000000,0.0,6.000000,0.000065,2,0.000065,1,0.0,0.0,0.000065,0
4,192.168.100.1->192.168.100.27,0,5897.666667,0.0,2.333333,0.000371333333333333,2,0.000371333333333333,1,0.0,0.0,0.000371,0


In [2]:
print(df.isnull().sum())

node_id                   0
proto_mean                0
seq_mean                  0
stddev_mean               0
N_IN_Conn_P_SrcIP_mean    0
min_mean                  0
state_number_mean         4
mean_mean                 0
N_IN_Conn_P_DstIP_mean    1
drate_mean                1
srate_mean                0
max_mean                  0
attack                    0
dtype: int64


In [3]:
df["state_number_mean"] = df["state_number_mean"].fillna(df["state_number_mean"].mode()[0])

In [4]:
df["N_IN_Conn_P_DstIP_mean"] = df["N_IN_Conn_P_DstIP_mean"].fillna(df["N_IN_Conn_P_DstIP_mean"].mode()[0])

In [5]:
df["drate_mean"] = df["drate_mean"].fillna(df["drate_mean"].mode()[0])

In [6]:
numeric_data = df.select_dtypes(include=np.number)
print(numeric_data.columns.tolist())
numeric_data.head()


['seq_mean', 'stddev_mean', 'N_IN_Conn_P_SrcIP_mean', 'drate_mean', 'srate_mean', 'max_mean', 'attack']


,seq_mean,stddev_mean,N_IN_Conn_P_SrcIP_mean,drate_mean,srate_mean,max_mean,attack
0,4521.000000,0.0,6.000000,0.0,0.0,0.000055,0
1,4280.000000,0.0,6.000000,0.0,0.0,0.000058,0
2,5412.000000,0.0,6.000000,0.0,0.0,0.000111,0
3,4287.000000,0.0,6.000000,0.0,0.0,0.000065,0
4,5897.666667,0.0,2.333333,0.0,0.0,0.000371,0


In [7]:
model = IsolationForest(
    contamination=0.05,
    random_state=42
)

# Predict anomalies
df["anomaly"] = model.fit_predict(numeric_data)

df["anomaly_label"] = df["anomaly"].map({
    1: "Normal",
    -1: "Anomaly"
})

df.head()


,node_id,proto_mean,seq_mean,stddev_mean,N_IN_Conn_P_SrcIP_mean,min_mean,state_number_mean,mean_mean,N_IN_Conn_P_DstIP_mean,drate_mean,srate_mean,max_mean,attack,anomaly,anomaly_label
0,192.168.100.1->192.168.100.147,0,4521.000000,0.0,6.000000,0.000055,2,0.000055,1,0.0,0.0,0.000055,0,1,Normal
1,192.168.100.1->192.168.100.148,0,4280.000000,0.0,6.000000,0.000058,2,0.000058,1,0.0,0.0,0.000058,0,1,Normal
2,192.168.100.1->192.168.100.149,0,5412.000000,0.0,6.000000,0.000111,2,0.000111,1,0.0,0.0,0.000111,0,1,Normal
3,192.168.100.1->192.168.100.150,0,4287.000000,0.0,6.000000,0.000065,2,0.000065,1,0.0,0.0,0.000065,0,1,Normal
4,192.168.100.1->192.168.100.27,0,5897.666667,0.0,2.333333,0.000371333333333333,2,0.000371333333333333,1,0.0,0.0,0.000371,0,1,Normal


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156 entries, 0 to 155
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   node_id                 156 non-null    object 
 1   proto_mean              156 non-null    object 
 2   seq_mean                156 non-null    float64
 3   stddev_mean             156 non-null    float64
 4   N_IN_Conn_P_SrcIP_mean  156 non-null    float64
 5   min_mean                156 non-null    object 
 6   state_number_mean       156 non-null    object 
 7   mean_mean               156 non-null    object 
 8   N_IN_Conn_P_DstIP_mean  156 non-null    object 
 9   drate_mean              156 non-null    float64
 10  srate_mean              156 non-null    float64
 11  max_mean                156 non-null    float64
 12  attack                  156 non-null    int64  
 13  anomaly                 156 non-null    int64  
 14  anomaly_label           156 non-null    ob

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA



In [13]:
X = numeric_data.drop(columns=["attack"])
y = df["attack"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [14]:
# Calculate anomaly score
df["anomaly_score"] = model.score_samples(numeric_data)

# Show anomaly score and label
df[["anomaly_score", "anomaly_label"]].head(50)

,anomaly_score,anomaly_label
0,-0.328333,Normal
1,-0.328478,Normal
2,-0.328780,Normal
3,-0.328724,Normal
4,-0.339368,Normal
5,-0.335998,Normal
6,-0.349872,Normal
7,-0.401949,Normal
8,-0.348741,Normal
9,-0.337074,Normal


In [15]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [16]:
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

In [18]:
x=X_train_scaled
y=df.attack
kf=KFold(n_splits=5)
kf.get_n_splits()
print(kf)
KFold(n_splits=2, random_state=None, shuffle=False)
for i, (train_index, test_index) in enumerate(kf.split(x)):
    print(f"Fold {i}:")
    print(f"Train:index={train_index}")
    print(f"Test:index={test_index}")

KFold(n_splits=5, random_state=None, shuffle=False)
Fold 0:
Train:index=[ 25  26  27  28  29  30  31  32  33  34  35  36  37  38  39  40  41  42
  43  44  45  46  47  48  49  50  51  52  53  54  55  56  57  58  59  60
  61  62  63  64  65  66  67  68  69  70  71  72  73  74  75  76  77  78
  79  80  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95  96
  97  98  99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114
 115 116 117 118 119 120 121 122 123]
Test:index=[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24]
Fold 1:
Train:index=[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  50  51  52  53  54  55  56  57  58  59  60
  61  62  63  64  65  66  67  68  69  70  71  72  73  74  75  76  77  78
  79  80  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95  96
  97  98  99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114
 115 116 117 118 119 120 121 122 123]
Test:index=[2

In [19]:
from sklearn.linear_model import LogisticRegression

In [24]:

x = X_train_scaled
y = y_train
lor = LogisticRegression()
print(cross_val_score(lor, x, y, cv=kf))

[0.72       0.8        0.8        0.84       0.91666667]
